# Inspect a trained model

This notebook uses the same test folders and helpers as `scripts/test_predict.py`
(the OSM **test** folder, not the training OSM folder). It shows the input stack
and a few prediction patches. The full paper figure set stays in the script:

```bash
poetry run python scripts/test_predict.py
```

`plot_input_pred_patch` also needs the Google satellite folder from `paths.yaml`.
It saves a JPEG and this notebook displays that file. Change `specific_batch` and
`specific_patch` to look at another chip.


In [ ]:
import os

import torch
from torch.utils.data import DataLoader
from torchgeo.datasets import BoundingBox, stack_samples
from torchgeo.datasets.splits import random_bbox_splitting
from torchgeo.samplers import GridGeoSampler
from lightning.pytorch import seed_everything
from IPython.display import Image, display

from ps_density.paths import load_paths
from ps_density.prediction_plots import plot_input_pred_patch
from ps_density.torchgeo_classes_def import PixelwiseRegressionTask
from ps_density.torchgeo_fun_to_read_input import (
    RasterDatasetOptical_imgs,
    RasterDataset_imgs,
    RasterDataset_msks,
    plot_input,
    scale_coh,
)

_cfg = load_paths()
experiment_dir = os.path.join(
    str(_cfg.outputs.revision_results_dir), _cfg.outputs.test_experiment_name
)
os.makedirs(experiment_dir, exist_ok=True)

imgs = RasterDataset_imgs(
    paths=str(_cfg.input.coherence_summer_dir), transforms=scale_coh
)
imgs_OSM = RasterDataset_imgs(paths=str(_cfg.input.osm_test_dir))
msks = RasterDataset_msks(paths=str(_cfg.input.mask_egms_dir))
dataset = (imgs & imgs_OSM) & msks
print("crs", dataset.crs, "res", dataset.res, "length", len(dataset))


In [ ]:
bounding_box = BoundingBox(
    minx=3.0,
    maxx=7.999999996,
    miny=50.000000004,
    maxy=54.0,
    mint=0.0,
    maxt=9.223372036854776e18,
)
sample = dataset[bounding_box]
print("input channels", sample["image"].shape[0])
print("mask values", torch.unique(sample["mask"]))
plot_input(sample, rows=1)


In [ ]:
seed = 54321
patch_size = 64
batch_size = 16
specific_batch = 0
specific_patch = 1

seed_everything(seed, workers=True)
generator = torch.Generator().manual_seed(seed)

model = PixelwiseRegressionTask.load_from_checkpoint(
    checkpoint_path=str(_cfg.checkpoints.netherlands_updated_osm)
)
model.eval()

_, _, test_dataset = random_bbox_splitting(
    dataset, [0.7, 0.15, 0.15], generator=generator
)
dataloader_pred = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    sampler=GridGeoSampler(test_dataset, patch_size, patch_size),
    num_workers=0,
    collate_fn=stack_samples,
    generator=generator,
)
opt = RasterDatasetOptical_imgs(paths=str(_cfg.input.google_satellite_dir))
outputdir_plots = os.path.join(
    str(_cfg.outputs.plots_review_dir), "notebook_test_pred"
)
os.makedirs(outputdir_plots, exist_ok=True)

df_mae = plot_input_pred_patch(
    dataloader_pred,
    model,
    opt,
    batch_size,
    True,
    True,
    True,
    outputdir_plots,
    specific_batch=specific_batch,
    specific_patch=specific_patch,
)
print(df_mae)
display(
    Image(
        filename=os.path.join(
            outputdir_plots,
            "model_pred_b{0}_{1}.jpg".format(specific_batch, specific_patch),
        )
    )
)


## Optional second patch

Edit the batch and patch indices, then run this cell.


In [ ]:
specific_batch_2 = 0
specific_patch_2 = 4

df_mae_2 = plot_input_pred_patch(
    dataloader_pred,
    model,
    opt,
    batch_size,
    True,
    False,
    True,
    outputdir_plots,
    specific_batch=specific_batch_2,
    specific_patch=specific_patch_2,
)
print(df_mae_2)
display(
    Image(
        filename=os.path.join(
            outputdir_plots,
            "model_pred_b{0}_{1}.jpg".format(specific_batch_2, specific_patch_2),
        )
    )
)


## Optional Rome transfer

Needs the Italy folders and the Rome Google satellite raster in `paths.yaml`.
This cell only draws one patch. The full Rome prediction GeoTIFF is written by
`scripts/test_predict.py`.


In [ ]:
rome = (
    RasterDataset_imgs(paths=str(_cfg.input.rome_coherence_dir), transforms=scale_coh)
    & RasterDataset_imgs(paths=str(_cfg.input.rome_osm_dir))
    & RasterDataset_msks(paths=str(_cfg.input.rome_mask_dir))
)
rome_batch = 0
rome_patch = 0
dataloader_rome = DataLoader(
    dataset=rome,
    batch_size=batch_size,
    sampler=GridGeoSampler(rome, patch_size, patch_size),
    num_workers=0,
    collate_fn=stack_samples,
    generator=generator,
)
opt_rome = RasterDatasetOptical_imgs(paths=str(_cfg.input.rome_google_satellite_dir))
rome_plot_dir = os.path.join(str(_cfg.outputs.plots_review_rome_dir), "notebook_rome")
os.makedirs(rome_plot_dir, exist_ok=True)
df_rome = plot_input_pred_patch(
    dataloader_rome,
    model,
    opt_rome,
    batch_size,
    True,
    True,
    True,
    rome_plot_dir,
    specific_batch=rome_batch,
    specific_patch=rome_patch,
)
print(df_rome)
display(
    Image(
        filename=os.path.join(
            rome_plot_dir, "model_pred_b{0}_{1}.jpg".format(rome_batch, rome_patch)
        )
    )
)
